In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
%pprint

Pretty printing has been turned OFF


In [4]:
dir = "../processed_data/"

In [5]:
def check_simulations(freq, res): 
    dat = []
    files = glob.glob(dir+res+"/"+res+"_"+freq+"_precip_*5x5.nc")
    for f in files: 
        ds = xr.open_dataset(f)
        v = f.split("/")[-1].split("_")[4]
        m = f.split("/")[-1].split("_")[3]
        
        d = pd.DataFrame({'freq': [freq], 'model': [m], 'variant': [v], 'length': [len(ds.time)], 
                          'start_date': [ds.time[0].values], 'end_date': [ds.time[-1].values], 
                         'grid_cells': [ds.pr.isel(time = 0).count().values]})
        dat.append(d)

    dat = pd.concat(dat)
    dat.to_csv(dir+res+"_simulation_table_"+freq+".csv", index = False)
    
    return()

In [6]:
check_simulations("mon", "cmip")

()

In [7]:
check_simulations("day", "cmip")

()

In [8]:
check_simulations("day", "highres-cmip")

()

In [9]:
check_simulations("mon", "highres-cmip")

()

# check simulations

### regular CMIP monthly:

In [10]:
mon_table = pd.read_csv(dir+"cmip_simulation_table_mon.csv")
mon_table = mon_table[(mon_table['start_date'] <= "1930-01")]

In [11]:
# remove high-res models if analyzing highres separately
# mon_table = mon_table[mon_table["model"] != "CNRM-CM6-1-HR"]

In [12]:
# check for missing dates:
mon_table[mon_table["length"] < 3012]

,freq,model,variant,length,start_date,end_date,grid_cells
11,mon,CAMS-CSM1-0,r1i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,708
26,mon,CAMS-CSM1-0,r2i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,708
99,mon,E3SM-1-0,r5i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,713
111,mon,IITM-ESM,r1i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,712
113,mon,E3SM-1-0,r4i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,713
125,mon,E3SM-1-0,r1i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,713
142,mon,E3SM-1-0,r2i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,713
163,mon,E3SM-1-0,r3i1p1f1,3000,1850-01-16 12:00:00,2099-12-16 12:00:00,713
277,mon,GISS-E2-1-G,r4i1p5f1,2412,1850-01-16 12:00:00,2050-12-16 12:00:00,706


In [13]:
mon_table[mon_table["grid_cells"] < 700]

,freq,model,variant,length,start_date,end_date,grid_cells
40,mon,NESM3,r2i1p1f1,3012,1850-01-16 12:00:00,2100-12-16 12:00:00,696
65,mon,NESM3,r1i1p1f1,3012,1850-01-16 12:00:00,2100-12-16 12:00:00,696


In [14]:
## number of simulations per model
mon_table.groupby("model").count()

,freq,variant,length,start_date,end_date,grid_cells
model,,,,,,
ACCESS-CM2,10,10,10,10,10,10
ACCESS-ESM1-5,40,40,40,40,40,40
AWI-CM-1-1-MR,1,1,1,1,1,1
BCC-CSM2-MR,1,1,1,1,1,1
CAMS-CSM1-0,2,2,2,2,2,2
CAS-ESM2-0,2,2,2,2,2,2
CESM2,3,3,3,3,3,3
CESM2-WACCM,3,3,3,3,3,3
CMCC-CM2-SR5,1,1,1,1,1,1


In [15]:
print("there are", len(mon_table), "monthly simulations")
print("there are", len(set(mon_table.model)), "models")

there are 346 monthly simulations
there are 44 models


In [16]:
## sort table before selecting single variant for analysis
mon_table = mon_table.sort_values("variant").reset_index()
mon_table = pd.concat([mon_table[mon_table.variant == "r1i1p1f1"], mon_table[mon_table.variant != "r1i1p1f1"]])
mon_table_onevar = mon_table.groupby("model").first().reset_index()

In [17]:
cmip_mon_sims = list(mon_table["model"]+"_"+mon_table["variant"])
cmip_mon_onevar = list(mon_table_onevar["model"]+"_"+mon_table_onevar["variant"])

### regular CMIP day:

In [18]:
day_table = pd.read_csv(dir+"cmip_simulation_table_day.csv")
day_table = day_table[(day_table['start_date'] <= "1950-01")]
# remove high-res models
#day_table = day_table[day_table["model"] != "CNRM-CM6-1-HR"]

In [19]:
day_table[day_table["grid_cells"] < 700]

,freq,model,variant,length,start_date,end_date,grid_cells
107,day,NESM3,r1i1p1f1,91676,1850-01-01 12:00:00,2100-12-31 12:00:00,696


In [20]:
day_table[day_table["length"] < 90360] #90360 is number of days 1850-2100 with 360-day calendar

,freq,model,variant,length,start_date,end_date,grid_cells
253,day,UKESM1-0-LL,r15i1p1f2,35640,1850-01-01 12:00:00,2099-12-30 12:00:00,710
260,day,UKESM1-0-LL,r14i1p1f2,80640,1850-01-01 12:00:00,2099-12-30 12:00:00,710


In [21]:
day_table = day_table[day_table["length"] >= 90360]

In [22]:
day_table.groupby("model").count()

,freq,variant,length,start_date,end_date,grid_cells
model,,,,,,
ACCESS-CM2,10,10,10,10,10,10
ACCESS-ESM1-5,40,40,40,40,40,40
BCC-CSM2-MR,1,1,1,1,1,1
CAMS-CSM1-0,1,1,1,1,1,1
CESM2,2,2,2,2,2,2
CESM2-WACCM,3,3,3,3,3,3
CMCC-CM2-SR5,1,1,1,1,1,1
CMCC-ESM2,1,1,1,1,1,1
CNRM-CM6-1,1,1,1,1,1,1


In [23]:
print("there are", len(day_table), "daily simulations")
print("there are", len(set(day_table.model)), "models")

there are 284 daily simulations
there are 37 models


In [24]:
day_table = day_table.sort_values("variant").reset_index()
day_table = pd.concat([day_table[day_table.variant == "r1i1p1f1"], day_table[day_table.variant != "r1i1p1f1"]])
day_table_onevar = day_table.groupby("model").first().reset_index()

In [25]:
cmip_day_sims = list(day_table["model"]+"_"+day_table["variant"])
cmip_day_onevar = list(day_table_onevar["model"]+"_"+day_table_onevar["variant"])

### HighRes mon:

In [26]:
hr_mon_table = pd.read_csv(dir+"highres-cmip_simulation_table_mon.csv")
## remove low-res models
hr_mon_table = hr_mon_table.loc[~hr_mon_table.model.isin(["CMCC-CM2-HR4", "CNRM-CM6-1", "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "MPI-ESM1-2-HR"])]
hr_mon_table

,freq,model,variant,length,start_date,end_date,grid_cells
0,mon,MPI-ESM1-2-XR,r1i1p1f1,1212,1950-01-31 23:58:00,2050-12-31 23:58:00,713
2,mon,EC-Earth3P-HR,r2i1p2f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,713
3,mon,HadGEM3-GC31-HM,r1i3p1f1,1212,1950-01-16 00:00:00,2050-12-16 00:00:00,713
4,mon,EC-Earth3P-HR,r3i1p2f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,713
7,mon,EC-Earth3P-HR,r1i1p2f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,713
9,mon,HiRAM-SIT-HR,r1i1p1f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,713
13,mon,FGOALS-f3-H,r1i1p1f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,709
14,mon,HiRAM-SIT-LR,r1i1p1f1,1212,1950-01-16 12:00:00,2050-12-16 12:00:00,713
15,mon,HadGEM3-GC31-HM,r1i1p1f1,1212,1950-01-16 00:00:00,2050-12-16 00:00:00,713
20,mon,HadGEM3-GC31-HM,r1i2p1f1,1212,1950-01-16 00:00:00,2050-12-16 00:00:00,713


In [27]:
print("there are", len(hr_mon_table), "HR monthly simulations")
print("there are", len(set(hr_mon_table.model)), "HR models")

there are 15 HR monthly simulations
there are 9 HR models


In [28]:
hr_mon_vars = list(hr_mon_table["model"]+"_"+hr_mon_table["variant"])

### HighRes day:

In [29]:
hr_day_table = pd.read_csv(dir+"highres-cmip_simulation_table_day.csv")
## remove low-res models
hr_day_table = hr_day_table.loc[~hr_day_table.model.isin(["CMCC-CM2-HR4", "CNRM-CM6-1", "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "MPI-ESM1-2-HR"])]
hr_day_table

,freq,model,variant,length,start_date,end_date,grid_cells
3,day,HadGEM3-GC31-HM,r1i1p1f1,36360,1950-01-01 12:00:00,2050-12-30 12:00:00,713
8,day,MPI-ESM1-2-XR,r1i1p1f1,36890,1950-01-01 23:58:00,2050-12-31 23:58:00,713
9,day,HadGEM3-GC31-HM,r1i3p1f1,36360,1950-01-01 12:00:00,2050-12-30 12:00:00,713
10,day,EC-Earth3P-HR,r1i1p2f1,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713
12,day,EC-Earth3P-HR,r2i1p2f1,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713
13,day,EC-Earth3P-HR,r3i1p2f1,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713
14,day,CNRM-CM6-1-HR,r3i1p1f2,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713
15,day,HadGEM3-GC31-HH,r1i1p1f1,36345,1950-01-16 12:00:00,2050-12-30 12:00:00,713
16,day,CNRM-CM6-1-HR,r2i1p1f2,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713
17,day,CNRM-CM6-1-HR,r1i1p1f2,36890,1950-01-01 12:00:00,2050-12-31 12:00:00,713


In [30]:
print("there are", len(hr_day_table), "HR daily simulations")
print("there are", len(set(hr_day_table.model)), "HR models")

there are 13 HR daily simulations
there are 7 HR models


In [31]:
hr_day_vars = list(hr_day_table["model"]+"_"+hr_day_table["variant"])

### Save list of model variants

In [32]:
import json

In [33]:
model_var_dict = {"cmip_mon": cmip_mon_sims, 
"cmip_mon_onevar": cmip_mon_onevar,
"cmip_day": cmip_day_sims, 
"cmip_day_onevar": cmip_day_onevar,    
"highres_mon": hr_mon_vars, 
                 "highres_day": hr_day_vars} 

In [34]:
json.dump(model_var_dict, open(dir+"model_var_dict.json", 'w' ))